In [ ]:
# 📦 Install required libraries
!pip install rdflib gradio pandas

In [ ]:
# 📚 Imports
from rdflib import Graph, Literal, RDF, URIRef, Namespace
import gradio as gr
import pandas as pd


In [ ]:
# 🌐 Define Namespaces and Graph
EX = Namespace("http://example.org/")
g = Graph()
g.bind("ex", EX)

# 🧠 Add educational data to the graph
g.add((EX.Alice, RDF.type, EX.Student))
g.add((EX.Alice, EX.enrolledIn, EX.MachineLearning))
g.add((EX.MachineLearning, RDF.type, EX.Course))
g.add((EX.MachineLearning, EX.hasTopic, Literal("Neural Networks")))
g.add((EX.MachineLearning, EX.hasTopic, Literal("Optimization")))

g.add((EX.Bob, RDF.type, EX.Professor))
g.add((EX.Bob, EX.teaches, EX.MachineLearning))


In [ ]:
# 🔍 Query Courses, Students, Professors
def query_course_relations(graph):
    query = '''
    PREFIX ex: <http://example.org/>
    SELECT ?student ?course ?professor WHERE {
        ?student a ex:Student .
        ?student ex:enrolledIn ?course .
        ?professor a ex:Professor .
        ?professor ex:teaches ?course .
    }
    '''
    return graph.query(query)


In [ ]:
# 📊 Return query result as formatted table
def format_course_query():
    rows = query_course_relations(g)
    results = [{"Student": row.student.split("/")[-1], "Course": row.course.split("/")[-1], "Professor": row.professor.split("/")[-1]} for row in rows]
    return pd.DataFrame(results)


In [ ]:
# 🎨 Gradio UI
with gr.Blocks() as demo:
    gr.Markdown("## 🎓 University Knowledge Graph Explorer")
    output = gr.Dataframe(label="Enrollment Data")
    btn = gr.Button("Run Query")
    btn.click(fn=format_course_query, outputs=output)

demo.launch()
